In [ ]:
%run robot_objects.ipynb
%run world_objects.ipynb

#### world

In [ ]:
# World class

from dataclasses import dataclass, field
from typing import Dict, List

# from src.main.world_objects.robot_objects.fuel_tank import FuelTank
# from src.main.world_objects.robot_objects.position import Position
# from src.main.world_objects.robot_objects.degrees import Degrees
# from src.main.world_objects.robot import Robot
# from src.main.world_objects.robot_objects.weapon import Weapon


@dataclass
class World:
    robots: Dict[str, Robot] = field(default_factory=dict)
    obstacles: Dict[Position, Obstacle] = field(default_factory=dict)

    def spawn_robot(
        self, name: str, position: Position, direction: Degrees, robot_type: str = "support", tank: 'FuelTank'= FuelTank()
    ) -> Robot:
        """Spawn a new robot with the given name, position, direction, and type."""
        name = name.lower()
        if name in self.robots:
            raise ValueError(f"A robot with the name '{name}' already exists.")
        self.robots[name] = Robot(
            name=name, position=position, direction=direction,robot_type=robot_type, tank=tank
        )
        return self.robots[name]

    def get_robot(self, name: str) -> Robot:
        """Get a robot by name."""
        name = name.lower()
        return self.robots.get(name)  #

    def _move_robot(self, name: str, steps: int, forward: bool) -> bool:
        """Move a robot by a certain number of steps."""
        robot = self.get_robot(name)
        print(robot.position, end=" --> ")
        if robot:
            update = robot.move_forward(steps) if forward else robot.move_back(steps)
            print(robot.position)
            return update
        return False

    def move_robot_forward(self, name: str, steps: int = 0):
        return self._move_robot(name=name, steps=steps, forward=True)
    def move_robot_backward(self, name: str, steps: int = 0):
        return self._move_robot(name=name, steps=steps, forward=False)

    def turn_robot_left(self, name: str, degrees: float = 90) -> None:
        """Turn a robot left by a certain number of degrees."""
        robot = self.get_robot(name)
        if robot:
            robot.turn_left(degrees)
            
    def turn_robot_right(self, name: str, degrees: float = 90) -> None:
        """Turn a robot right by a certain number of degrees."""
        robot = self.get_robot(name)
        if robot:
            robot.turn_right(degrees)

    def shoot(self, shooter_name: str) -> None:
        shooter = self.get_robot(shooter_name)
        if not shooter or shooter.weapon.ammo <= 0:
            print(f"{shooter_name} cannot shoot: Out of ammo.")

        # Update ammo and set status
        gun = shooter.weapon
        shooter.weapon = Weapon(gun.ammo_max,gun.ammo-1, gun.damage, gun.loading, gun.load_delay)
        # shooter.set_status(f"{shooter.name} fired weapon.")

        shooter_x, shooter_y = shooter.position.x, shooter.position.y
        shot_direction = shooter.direction
        # targets_hit = []

        for bot in self.robots.values():
            if bot.name == shooter_name:
                continue

            bot_x, bot_y = bot.position.x, bot.position.y

            # Check if bot is in the line of fire
            is_in_line_of_fire = (
                    (bot_y == shooter_y and
                     ((bot_x < shooter_x and shot_direction == Degrees(180)) or
                      (bot_x > shooter_x and shot_direction == Degrees(0)))) or
                    (bot_x == shooter_x and
                     ((bot_y < shooter_y and shot_direction == Degrees(270)) or
                      (bot_y > shooter_y and shot_direction == Degrees(90))))
            )

            if is_in_line_of_fire:
                bot.damage_shield(gun.damage)  # Apply damage, adjust as needed
                # targets_hit.append(bot.name)
                # bot.set_status(f"{bot.name} was shot.")
                return


    def __str__(self):
        return f"World:\nrobots: {', '.join(list(world.robots.keys()))}\nobstacles: {', '.join(list(self.obstacles.keys()))}"


In [ ]:
world = World()
print(world)
world

In [ ]:
bot = world.spawn_robot("bot", Position(0,0), Degrees(90))
print(bot)
bot

In [ ]:
print(world)
world

In [ ]:
bot1 = world.spawn_robot("bot1", Position(0,0), Degrees(90))
print(bot1)
bot1

In [ ]:
world.robots

In [ ]:
print("bot x =", bot.position.x, ", bot1 x =", bot1.position.x)
print("same x ? : ", bot.position.x == bot1.position.x)
print("how much is the diff?:",abs(bot.position.x - bot1.position.x))
print("which bot has a greater number?", ("bot" if bot.position.x > bot1.position.x else ("bot1" if bot1.position.x > bot.position.x else "they equal")))

print()
print("bot y =", bot.position.y,", bot1 y =", bot1.position.y)
print("same y ? : ", bot.position.y == bot1.position.y)
print("how much is the diff?:", abs(bot.position.y - bot1.position.y))
print("which bot has a greater number?", ("bot" if bot.position.y > bot1.position.y else ("bot1" if bot1.position.y > bot.position.y else "they equal")))


# could i abstract bot.position.x to bot.x? i feel like its only bot that should know that it 1st has to go thru position to get x but maybe not?
# somehow they arent in the same place?!, but they are still very close

In [ ]:
world.move_robot_forward("bot")

In [ ]:
world.move_robot_forward("bot", 1)

In [ ]:
world.move_robot_back("bot", 1)

In [ ]:
[cyborg.position for _, cyborg in world.robots.items()] # bots should not be able to hold the same position at the same time even at spawning 

In [ ]:
world.move_robot_forward("bot1", 100)
print(world.move_robot_back("bot1", 100))
world.move_robot_back("bot1", 1)
# there should have been a collision at least once

In [ ]:
bot.move_forward(5)
bot1.move_forward(5)
# expected shared position error

In [ ]:
[robot.position for _, robot in world.robots.items()]

In [ ]:
[robot.direction for _, robot in world.robots.items()]

In [ ]:
world.spawn_robot("bot")  # should provide random defualts (position&degrees)

In [ ]:
world.spawn_robot("bot", Position(0,0)) # how to expect required positional argumnt error anyway

In [ ]:
world.spawn_robot("bot", Position(0,0), Degrees(90)) # should append a value after the name so as not fail (fail silently idk())